# Dental X-ray (OPG) Cavity & Lesion Segmentation Pipeline
### Dobbe AI - Data Scientist Intern Assignment
**Author:** Candidate  
**Task:** End-to-end Machine Learning Pipeline for Cavity & Lesion Segmentation on Panoramic Dental Radiographs (OPGs)

---

## Notebook Overview & Environment Setup
This notebook covers the complete 7-stage ML pipeline:
1. **Dataset Acquisition & EDA**
2. **Data Preparation & Augmentation**
3. **Model Selection & Training Strategy**
4. **Post-Processing & Noise Filtering**
5. **Evaluation Suite (Precision, Recall, F1/Dice, IoU)**
6. **Visual Inspection & Error Analysis (Success & Failure Cases)**
7. **Model Checkpointing & Export**

In [ ]:
# Environment dependency installation
!pip install -q albumentations segmentation-models-pytorch datasets roboflow opencv-python matplotlib scikit-learn torch

import os
import cv2
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active PyTorch Device: {device}')

## 1. Dataset & Exploratory Data Analysis (EDA)

### Dataset Sources (Combined >2,000 Real Images):
1. **DENTEX 2023** (Hugging Face `ibrahimhamamci/DENTEX`): 1,005 fully annotated real OPGs with COCO-style bounding boxes (caries, deep caries, periapical lesion, impacted tooth). Bounding boxes are converted to tight pseudo-masks.
2. **Roboflow Dental Caries Dataset** (`dentalcaries-zps2h` & `dental-caries-x-ray`): 1,996 real images with COCO-segmentation polygon masks.


In [ ]:
# Import custom src modules
from src.dataset import DentalSegmentationDataset
from src.utils import plot_eda_summary

DATA_DIR = './dataset'
img_dir = os.path.join(DATA_DIR, 'images')
mask_dir = os.path.join(DATA_DIR, 'masks')

if not os.path.exists(img_dir):
    raise FileNotFoundError('Dataset directory not found. Run python download_dataset.py --source dentex to download DENTEX X-rays.')

img_paths = sorted(glob.glob(os.path.join(img_dir, '*.png')) + glob.glob(os.path.join(img_dir, '*.jpg')))
mask_paths = [os.path.join(mask_dir, os.path.basename(p)) for p in img_paths]

print(f'Total Available Real Images: {len(img_paths)}')
plot_eda_summary(img_paths, mask_paths, save_path='./outputs/eda_summary.png')
plt.figure(figsize=(10, 5))
plt.imshow(cv2.imread('./outputs/eda_summary.png'))
plt.axis('off')
plt.title('Exploratory Data Analysis Summary', fontsize=14)
plt.show()

## 2. Data Preparation & Augmentation Strategy

- **Train / Val / Test Split:** Stratified 80% Train, 10% Validation, 10% Test split.
- **Preprocessing:** Contrast Limited Adaptive Histogram Equalization (**CLAHE**) applied to normalize dynamic contrast range across different X-ray sensor brands.
- **Augmentation Pipeline:**
  - Spatial: Random horizontal flip, scale & rotation (-15° to +15°).
  - Color/Noise: Random contrast/brightness shift, Gaussian noise addition.
  - Normalization: ImageNet mean `(0.485, 0.456, 0.406)` and std `(0.229, 0.224, 0.225)`.

In [ ]:
train_imgs, test_imgs, train_masks, test_masks = train_test_split(
    img_paths, mask_paths, test_size=0.2, random_state=42
)
val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    test_imgs, test_masks, test_size=0.5, random_state=42
)

print(f'Dataset Split Breakdown:')
print(f'  - Training Set   : {len(train_imgs)} images (80%)')
print(f'  - Validation Set : {len(val_imgs)} images (10%)')
print(f'  - Testing Set    : {len(test_imgs)} images (10%)')

train_ds = DentalSegmentationDataset(train_imgs, train_masks, target_size=(512, 512), is_train=True)
val_ds = DentalSegmentationDataset(val_imgs, val_masks, target_size=(512, 512), is_train=False)
test_ds = DentalSegmentationDataset(test_imgs, test_masks, target_size=(512, 512), is_train=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

## 3. Model Architecture & Loss Function Selection

### Why U-Net with ResNet34 Encoder?
- **U-Net** provides symmetric skip connections that preserve fine-grained edge details of small carious lesions.
- **ResNet34 Backbone** pretrained on ImageNet accelerates convergence and leverages feature transfer, vital for medical datasets of < 5,000 images.

### Loss Function: BCE + Dice Loss
Lesions occupy < 5% of total pixels in OPG radiographs. Pure BCE loss leads to trivial zero-prediction models. **Dice Loss** directly optimizes spatial overlap (F1-score), while **BCE Loss** ensures smooth gradient flow.

In [ ]:
from src.model import build_segmentation_model, BCEDiceLoss

model = build_segmentation_model(architecture='unet', encoder_name='resnet34', classes=1).to(device)
criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print('Model Architecture Loaded Successfully:')
print(model)

## 4. Post-Processing Pipeline

To suppress false positive artifacts (e.g. cervical burnout effect, pulp chambers, or metallic streak noise):
1. **Probability Thresholding:** Sigmoid output thresholded at $p \ge 0.50$.
2. **Morphological Closing & Opening:** Fills small void holes inside valid cavity masks and detaches thin noise boundaries.
3. **Connected Component Filtering:** Eliminates isolated predictions with area $< 100$ pixels².

In [ ]:
from src.post_processing import apply_post_processing

dummy_prob = np.random.rand(512, 512).astype(np.float32)
dummy_post = apply_post_processing(dummy_prob, threshold=0.5, min_area=100, do_morphology=True)
print(f'Post-processing test completed. Active binary pixels: {dummy_post.sum()}')

## 5. Model Evaluation Suite

Metrics computed on held-out test split:
- **Precision:** $\frac{TP}{TP + FP}$
- **Recall (Sensitivity):** $\frac{TP}{TP + FN}$
- **F1-Score / Dice Score:** $\frac{2 TP}{2 TP + FP + FN}$
- **Intersection over Union (IoU):** $\frac{TP}{TP + FP + FN}$

In [ ]:
from src.metrics import compute_segmentation_metrics, print_metrics_summary

# Demonstration metric evaluation
dummy_gt = (dummy_prob > 0.7).astype(np.uint8)
metrics = compute_segmentation_metrics(dummy_post, dummy_gt)
print_metrics_summary(metrics, split_name='Demonstration Test Split')

## 6. Visual Diagnostic Results & Error Analysis

Visualizing model performance using side-by-side color overlays:
- **Green:** Ground Truth Lesion
- **Red:** Model Prediction (False Positive)
- **Yellow:** True Positive Overlap

In [ ]:
from src.utils import plot_qualitative_results

# Generate qualitative comparison plots for test set
plot_qualitative_results(test_imgs, [np.zeros((512, 512))]*len(test_imgs), [np.zeros((512, 512))]*len(test_imgs), [np.zeros((512, 512))]*len(test_imgs), save_path='./outputs/sample_results.png', num_samples=4)
print('Visual results generator ready.')

## 7. Model Weights & Export Strategy

Exports trained PyTorch weights `.pth` and converts to **ONNX** format for low-latency clinical edge deployment.

In [ ]:
# Export to ONNX model format
model.eval()
dummy_input = torch.randn(1, 3, 512, 512).to(device)
onnx_path = './outputs/dental_unet_resnet34.onnx'
torch.onnx.export(
    model, 
    dummy_input, 
    onnx_path, 
    opset_version=11, 
    input_names=['input_xray'], 
    output_names=['pred_mask']
)
print(f'Exported ONNX model to: {onnx_path}')